## Preparação do ambiente

In [3]:
!pip install -q pyspark duckdb

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
import duckdb

## Obtenção do conjunto de dados


In [ ]:
!mkdir -p dataset/zip/

In [ ]:
!curl -L -o /content/dataset/zip/animal-crossing-new-horizons-nookplaza-dataset.zip https://www.kaggle.com/api/v1/datasets/download/jessicali9530/animal-crossing-new-horizons-nookplaza-dataset

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  576k  100  576k    0     0  1563k      0 --:--:-- --:--:-- --:--:-- 1563k


In [ ]:
!unzip /content/dataset/zip/animal-crossing-new-horizons-nookplaza-dataset.zip -d /content/dataset/

Archive:  /content/dataset/zip/animal-crossing-new-horizons-nookplaza-dataset.zip
  inflating: /content/dataset/accessories.csv  
  inflating: /content/dataset/achievements.csv  
  inflating: /content/dataset/art.csv  
  inflating: /content/dataset/bags.csv  
  inflating: /content/dataset/bottoms.csv  
  inflating: /content/dataset/construction.csv  
  inflating: /content/dataset/dress-up.csv  
  inflating: /content/dataset/fencing.csv  
  inflating: /content/dataset/fish.csv  
  inflating: /content/dataset/floors.csv  
  inflating: /content/dataset/fossils.csv  
  inflating: /content/dataset/headwear.csv  
  inflating: /content/dataset/housewares.csv  
  inflating: /content/dataset/insects.csv  
  inflating: /content/dataset/miscellaneous.csv  
  inflating: /content/dataset/music.csv  
  inflating: /content/dataset/other.csv  
  inflating: /content/dataset/photos.csv  
  inflating: /content/dataset/posters.csv  
  inflating: /content/dataset/reactions.csv  
  inflating: /content/datas

## Criando uma spark session

In [5]:
spark = SparkSession.builder \
    .appName("cozy-etl") \
    .master("local[*]") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

## Criando nossa camada bronze

Ingestão dos csvs

In [6]:
acessorios = spark.read.csv("/content/dataset/accessories.csv", header=True, inferSchema=True)
conquistas = spark.read.csv("/content/dataset/achievements.csv", header=True, inferSchema=True)
bolsas = spark.read.csv("/content/dataset/bags.csv", header=True, inferSchema=True)
vestimentas = spark.read.csv("/content/dataset/dress-up.csv", header=True, inferSchema=True)
artes = spark.read.csv("/content/dataset/art.csv", header=True, inferSchema=True)
cabeca = spark.read.csv("/content/dataset/headwear.csv", header=True, inferSchema=True)
calca = spark.read.csv("/content/dataset/bottoms.csv", header=True, inferSchema=True)
musica = spark.read.csv("/content/dataset/music.csv", header=True, inferSchema=True)
personagens = spark.read.csv("/content/dataset/villagers.csv", header=True, inferSchema=True)
ferramentas = spark.read.csv("/content/dataset/tools.csv", header=True, inferSchema=True)
receitas = spark.read.csv("/content/dataset/recipes.csv", header=True, inferSchema=True)

Adição de colunas de controle



In [7]:
acessorios = acessorios.withColumn('dt_ingestao', F.current_timestamp())
conquistas = conquistas.withColumn('dt_ingestao', F.current_timestamp())
bolsas = bolsas.withColumn('dt_ingestao', F.current_timestamp())
vestimentas = vestimentas.withColumn('dt_ingestao', F.current_timestamp())
artes = artes.withColumn('dt_ingestao', F.current_timestamp())
cabeca = cabeca.withColumn('dt_ingestao', F.current_timestamp())
calca = calca.withColumn('dt_ingestao', F.current_timestamp())
musica = musica.withColumn('dt_ingestao', F.current_timestamp())
personagens = personagens.withColumn('dt_ingestao', F.current_timestamp())
ferramentas = ferramentas.withColumn('dt_ingestao', F.current_timestamp())
receitas = receitas.withColumn('dt_ingestao', F.current_timestamp())

In [9]:
acessorios.write.parquet("/content/lakehouse/bronze/acessorios")
conquistas.write.parquet("/content/lakehouse/bronze/conquistas")
bolsas.write.parquet("/content/lakehouse/bronze/bolsas")
vestimentas.write.parquet("/content/lakehouse/bronze/vestimentas")
artes.write.parquet("/content/lakehouse/bronze/artes")
cabeca.write.parquet("/content/lakehouse/bronze/cabeca")
calca.write.parquet("/content/lakehouse/bronze/calca")
musica.write.parquet("/content/lakehouse/bronze/musica")
personagens.write.parquet("/content/lakehouse/bronze/personagens")
ferramentas.write.parquet("/content/lakehouse/bronze/ferramentas")
receitas.write.parquet("/content/lakehouse/bronze/receitas")

Adicionando uma camada de persistência



In [11]:
con = duckdb.connect("cozy-lakehouse.db")

In [16]:
con.execute("""CREATE SCHEMA IF NOT EXISTS bronze""")

In [17]:
con.execute("""CREATE TABLE bronze.acessorios AS
SELECT *
FROM '/content/lakehouse/bronze/acessorios/*.parquet'""")
con.execute("""CREATE TABLE bronze.conquistas AS
SELECT *
FROM '/content/lakehouse/bronze/conquistas/*.parquet'""")
con.execute("""CREATE TABLE bronze.bolsas AS
SELECT *
FROM '/content/lakehouse/bronze/bolsas/*.parquet'""")
con.execute("""CREATE TABLE bronze.vestimentas AS
SELECT *
FROM '/content/lakehouse/bronze/vestimentas/*.parquet'""")
con.execute("""CREATE TABLE bronze.artes AS
SELECT *
FROM '/content/lakehouse/bronze/artes/*.parquet'""")
con.execute("""CREATE TABLE bronze.cabeca AS
SELECT *
FROM '/content/lakehouse/bronze/cabeca/*.parquet'""")
con.execute("""CREATE TABLE bronze.calca AS
SELECT *
FROM '/content/lakehouse/bronze/calca/*.parquet'""")
con.execute("""CREATE TABLE bronze.musica AS
SELECT *
FROM '/content/lakehouse/bronze/musica/*.parquet'""")
con.execute("""CREATE TABLE bronze.personagens AS
SELECT *
FROM '/content/lakehouse/bronze/personagens/*.parquet'""")
con.execute("""CREATE TABLE bronze.ferramentas AS
SELECT *
FROM '/content/lakehouse/bronze/ferramentas/*.parquet'""")
con.execute("""CREATE TABLE bronze.receitas AS
SELECT *
FROM '/content/lakehouse/bronze/receitas/*.parquet'""")

In [ ]:
con.sql("SELECT * FROM bronze.acessorios LIMIT 5").show()

In [ ]:
con.sql("SELECT * FROM bronze.cabeca LIMIT 5").show()

## Refinando a qualidade dos dados - camada silver

In [ ]:
con.execute("""CREATE SCHEMA IF NOT EXISTS silver""")

### Agrupamento de semelhantes

Busca de semelhantes

In [ ]:
con.sql("""DESCRIBE bronze.acessorios""").show()

Agrupamento das tabelas relacionadas

In [ ]:
con.execute(
    """
    CREATE TABLE silver.vestuario AS
    WITH agrupamento AS (
      SELECT
        *
      FROM bronze.vestimentas
      UNION ALL
      SELECT
        *
      FROM bronze.bolsas
      UNION ALL
      SELECT
        *
      FROM bronze.calca
      UNION ALL
      SELECT
        *
      FROM bronze.acessorios
      UNION ALL
      SELECT
        *
      FROM bronze.cabeca
      UNION ALL
      SELECT
        *
      FROM bronze.vestimentas
    )
    SELECT
      "Name" as nm_item,
      "Variation" as var_item,
      "Buy" as vl_compra,
      "Sell" as vl_venda,
      "Category" as ct_item,
      CASE WHEN
        "Color1" = "Color2"
        THEN "Color1"
        ELSE "Color1" || ', ' || "Color2"
      END as nm_cores_disp,
      "Source" as nm_fonte,
      "Seasonal Availability" as disp_sazonal,
      "Style" as estilo,
      "Lable Themes" as categorias,
      dt_ingestao
    FROM agrupamento
    """
)

Padronização de colunas

In [ ]:
con.execute(
    """
    CREATE TABLE silver.obras_arte AS
    SELECT
      *
    FROM bronze.artes
    """
)

## Preparando nossos dados para análise - camada gold

In [ ]:
con.execute("""CREATE SCHEMA IF NOT EXISTS gold""")

Catálogo de produtos de verão das Able Sisters

In [ ]:
con.execute(
    """
    """
)

Curadoria de obras de arte

In [ ]:
con.execute(
    """
    """
)

## Recap do que fizemos

In [ ]:
con.sql("SELECT schema_name FROM duckdb_schemas()").show()

In [ ]:
con.sql("SELECT table_name FROM duckdb_tables() WHERE schema_name = 'bronze'").show()